In [10]:
import pandas as pd

data = pd.read_csv("../data/Perioperative_Dataset_csv.csv")

data = data.drop(columns = ['CASEDATE', 'Unnamed: 0'])

timestamp_columns = [ 'ADMIT_DATE_TIME', 'DISCHARGE_DATE_TIME', 'CLEANINGENDDTTM',
       'CLEANINGSTARTDTTM', 'DRESSINGDTTM.1', 'INCISIONDTTM.1',
       'MAINSTARTINDUCTION.1', 'PATIENTARRIVEDDTTM', 'PATIENTINDTTM.1',
       'PATIENTREADYFORDTTM', 'PATIENTSENTDTTM', 'PREPSTARTDTTM',
       'RECOVERYINDTTM.1', 'RECOVERYOUTDTTM.1', 'TIMEOUTDTTM.1']

for col in timestamp_columns:
    data[col] = pd.to_datetime(data[col], errors='coerce')

data = data[data['INCISIONDTTM.1'].notna()]

for col in timestamp_columns:
    if col != 'ADMIT_DATE_TIME': 
        data = data[data[col].isna() | (data['ADMIT_DATE_TIME'] < data[col])]

for col in timestamp_columns:
    if col != 'DISCHARGE_DATE_TIME':
        data = data[data[col].isna() | (data[col] < data['DISCHARGE_DATE_TIME'])]

(data['DISCHARGE_DATE_TIME'] - data['ADMIT_DATE_TIME']).dt.days.max()

count = (data['ADMIT_DATE_TIME'] < '2009-01-01').sum()

data = data[data['ADMIT_DATE_TIME'] >= '2009-01-01']

(data['DISCHARGE_DATE_TIME'] - data['ADMIT_DATE_TIME']).dt.days.max()

mapping = {
    'GEN': 'GENERAL_SURGERY', 'SUR': 'GENERAL_SURGERY',
    'GYN': 'OBGYN', 'OBS': 'OBGYN',
    'ORTHO': 'ORTHOPEDICS', 
    'NEURO': 'NEUROSURGERY',
    'CARDIAC': 'CARDIAC', 'CARDIO': 'CARDIAC',
    'URO': 'UROLOGY', 'UROL': 'UROLOGY',
    'OTO': 'ENT', 'ENT': 'ENT',
    'PAEDS': 'PEDIATRICS', 'PAED': 'PEDIATRICS',
    'VASC': 'VASCULAR',
    'PLASTI': 'PLASTIC_SURGERY', 'PASLTI': 'PLASTIC_SURGERY',
    'BREAST': 'BREAST', 'MAMA': 'BREAST',
    'OPHTHY': 'OPHTHALMOLOGY', 'OPTHAL': 'OPHTHALMOLOGY',
    'DENTAL': 'DENTAL_ORAL', 'ORAL': 'DENTAL_ORAL',
    'ONCO': 'ONCOLOGY', 
    'THORACIC': 'THORACIC',
    'GASTRO': 'GASTRO',
    'MED': 'MEDICAL', 'NEPHROLOGY': 'MEDICAL',
    'LITHO': 'INTERVENTIONAL', 'CATHLAB': 'INTERVENTIONAL'
}

data['SPECIALTY'] = data['SPECIALTY_1'].replace(mapping)

data.loc[
    data['SPECIALTY'].isna() &
    data['PROCEDUREDESCR_1'].str.contains(
        r'C SECTION|CESAREAN|DILAT|EVACUATION|HYSTERECTOMY|TUBAL|MYOMECTOMY|CERCLAGE|OVARIAN|PLACENTA|DYE TEST|BARTHOLIN|M.T.X',
        na=False
    ),
    'SPECIALTY'
] = 'OBGYN'

data.loc[
    data['SPECIALTY'].isna() &
    data['PROCEDUREDESCR_1'].str.contains(
        r'CHOLECYST|APPENDECTOMY|LAPAROTOMY|WOUND|DEBRID|HERNIA|THYROID|BIOPSY|LIPOMA|COLOSTOMY|GASTROSTOMY|SPLENECTOMY|HEMICOLECTOMY|HEPATECTOMY',
        na=False
    ),
    'SPECIALTY'
] = 'GENERAL_SURGERY'

data.loc[
    data['SPECIALTY'].isna() &
    data['PROCEDUREDESCR_1'].str.contains(
        r'FRACTURE|ORIF|KNEE|HIP|LAMINECTOMY|DISCECTOMY|AMPUTATION|FASCIOTOMY|ARTHRO|BONE|NAILING|FIXATION|TIBIA|FEMUR',
        na=False
    ),
    'SPECIALTY'
] = 'ORTHOPEDICS'

data.loc[
    data['SPECIALTY'].isna() &
    data['PROCEDUREDESCR_1'].str.contains(
        r'CRANI|VP SHUNT|BURR HOLE|BRAIN|SPINAL|LAMINECTOMY|CRANIECTOMY|NEURO',
        na=False
    ),
    'SPECIALTY'
] = 'NEUROSURGERY'

data.loc[
    data['SPECIALTY'].isna() &
    data['PROCEDUREDESCR_1'].str.contains(
        r'CABG|VALVE|VSD|ASD|TETRALOGY|CARDIAC|PACEMAKER|SHUNT|STERNOTOMY',
        na=False
    ),
    'SPECIALTY'
] = 'CARDIAC'

data.loc[
    data['SPECIALTY'].isna() &
    data['PROCEDUREDESCR_1'].str.contains(
        r'TURP|CYSTOSCOPY|NEPHRECTOMY|URETER|PCNL|STENT|BLADDER|HYDROCELE|HYPO',
        na=False
    ),
    'SPECIALTY'
] = 'UROLOGY'

data.loc[
    data['SPECIALTY'].isna() &
    data['PROCEDUREDESCR_1'].str.contains(
        r'TRACHEO|FESS|TONSIL|PAROTID|LARYNG|MASTOID|SINUS|SEPTOPLASTY',
        na=False
    ),
    'SPECIALTY'
] = 'ENT'

data.loc[
    data['SPECIALTY'].isna() &
    data['PROCEDUREDESCR_1'].str.contains(
        r'BRONCH|THORACOTOMY|LOBECTOMY|DECORTICATION|VATS|ESOPHAGECTOMY|LUNG',
        na=False
    ),
    'SPECIALTY'
] = 'THORACIC'

data.loc[
    data['SPECIALTY'].isna() &
    data['PROCEDUREDESCR_1'].str.contains(
        r'ARTRIO|BRACHIO|BASILIC|ENDARTERECTOMY|FISTULA|VERICOSE|VARICOAE|EMBOLECTOMY|PERITONEAL|DIALYSIS|AMPUTATION|THROMBECTOMY|LIGATION|FEMORAL|PORTA CATH',
         na=False
    ),
    'SPECIALTY'
] = 'VASCULAR'

data.loc[
    data['SPECIALTY'].isna() &
    data['PROCEDUREDESCR_1'].str.contains(
        r'FLAP COVERAGE|SPLIT THICKNESS SKIN GRAFT|REPAIR OF FACIAL LACERATION|CLEFT PALATE REPAIR|ROTATION FLAP|REPAIR OF HAND / FOREARM LACERATIONS - BONE GRAFT|LAT DORSI FLAP|FULL THICKNESS SKIN GRAFTING|EXCISION OF SCAR|SKIN GRAFTING|LOCAL FLAP|PECTORALIS MAJOR MYOCUTANEOUS FLAP|ABDOMINOPLASTY|CLEFT LIP REPAIR|FREE FLAP|EXCISION OF HEAMANGIOMA|RELEASE OF CONTRACTURE|DEBRIDEMENT OF BED SORE|REPAIR OF LIP LACERATION|REPAIR OF TONGUE LACERATION|ADVANCEMENT OF FLAP',
         na=False
    ),
    'SPECIALTY'
] = 'PLASTIC_SURGERY'


data = data.dropna(subset=['SPECIALTY'])


data['ID_seq'] = data['ID_NO_V'] + data['ENCRYPTED_MRNUMBER'] + data['ENCRYPTED_VISITID']

# ──────────────────────────────────────────────────────────────────────────────
# SECTION 1 — activity metadata (same as your original code)
# ──────────────────────────────────────────────────────────────────────────────

activity_mapping = {
    'ADMIT_DATE_TIME':       'Admission',
    'DISCHARGE_DATE_TIME':   'Discharge',
    'CLEANINGENDDTTM':       'Cleaning End',
    'CLEANINGSTARTDTTM':     'Cleaning Start',
    'DRESSINGDTTM.1':        'Dressing',
    'INCISIONDTTM.1':        'Incision',
    'MAINSTARTINDUCTION.1':  'Induction Start',
    'PATIENTARRIVEDDTTM':    'Patient Arrived',
    'PATIENTINDTTM.1':       'Patient In',
    'PATIENTREADYFORDTTM':   'Patient Ready',
    'PATIENTSENTDTTM':       'Patient Sent',
    'PREPSTARTDTTM':         'Prep Start',
    'RECOVERYINDTTM.1':      'Recovery In',
    'RECOVERYOUTDTTM.1':     'Recovery Out',
    'TIMEOUTDTTM.1':         'Timeout',
}

activity_order = {
    'Admission':      1,
    'Patient Arrived': 2,
    'Patient Sent':   3,
    'Cleaning Start': 4,
    'Cleaning End':   5,
    'Patient In':     6,
    'Induction Start': 7,
    'Patient Ready':  8,
    'Prep Start':     9,
    'Timeout':       10,
    'Incision':      11,
    'Dressing':      12,
    'Recovery In':   13,
    'Recovery Out':  14,
    'Discharge':     15,
}

# For All Data

df_melt = data.melt(
    id_vars=['ID_seq'],
    value_vars=timestamp_columns,
    var_name='activity',
    value_name='timestamp'
)


df_melt = df_melt.dropna(subset=['timestamp'])

df_melt['activity'] = df_melt['activity'].replace(activity_mapping)
df_melt['activity_order'] = df_melt['activity'].map(activity_order)

df_eventlog = df_melt.sort_values(by=['ID_seq', 'timestamp', 'activity_order'])

import pm4py

event_log = pm4py.format_dataframe(
    df_eventlog,
    case_id='ID_seq',
    activity_key='activity',
    timestamp_key='timestamp'
)

import pandas as pd
import numpy as np
from collections import defaultdict
from graphviz import Digraph

# ──────────────────────────────────────────────────────────────────────────────
# SECTION 2 — domain-prior weights
# Ask your domain experts to review / adjust these.
# The value of alpha_prior for each pair encodes "prior belief that A→B exists".
# Pairs not listed fall back to the global default (alpha=1, beta=1 → uniform).
# ──────────────────────────────────────────────────────────────────────────────

# Format: (A, B) -> (alpha_prior, beta_prior)
# alpha > 1  : believe this link exists
# beta  > 1  : believe this link is rare / weak
# (1, 1)     : completely neutral (uniform prior) — safe default
DOMAIN_PRIORS = {
    # Near-deterministic clinical sequences  → high alpha, low beta
    ('Admission',       'Patient Sent'):      (5.0, 1.0),
    ('Patient Sent',    'Patient Arrived'):   (5.0, 1.0),
    ('Patient In',      'Patient Ready'):     (5.0, 1.0),
    ('Patient Ready',   'Prep Start'):        (5.0, 1.0),
    ('Prep Start',      'Incision'):          (5.0, 1.0),
    ('Incision',        'Dressing'):          (5.0, 1.0),
    ('Dressing',        'Timeout'):           (5.0, 1.0),
    ('Timeout',         'Recovery In'):       (5.0, 1.0),
    ('Recovery In',     'Recovery Out'):      (5.0, 1.0),
    ('Recovery Out',    'Discharge'):         (5.0, 1.0),
    # Cleaning sub-process (optional, lower certainty)
    ('Cleaning Start',  'Cleaning End'):      (3.0, 1.0),
    ('Cleaning End',    'Patient In'):        (3.0, 1.0),
    # Rare but clinically important
    ('Induction Start', 'Patient Ready'):     (2.0, 1.0),
}

DEFAULT_ALPHA = 1.0   # ← ask domain experts; 1.0 = uniform / uninformative
DEFAULT_BETA  = 1.0
# ──────────────────────────────────────────────────────────────────────────────
# SECTION 3 — core Bayesian dependency functions
# ──────────────────────────────────────────────────────────────────────────────

def get_prior(a: str, b: str) -> tuple:
    """Return (alpha, beta) prior for the pair (a→b)."""
    return DOMAIN_PRIORS.get((a, b), (DEFAULT_ALPHA, DEFAULT_BETA))


def posterior_mean(f_ab: int, f_ba: int, alpha: float, beta: float) -> float:
    """
    E[p(A→B)] = (f(A→B) + alpha) / (f(A→B) + f(B→A) + alpha + beta)
    n_A  =  total transitions *out of A that went to B or back from B*
            (here approximated as f_ab + f_ba, matching your document formula)
    """
    n = f_ab + f_ba
    return (f_ab + alpha) / (n + alpha + beta)


def bayesian_dependency(a: str, b: str, f_ab: int, f_ba: int) -> float:
    """
    dep_bayes(A,B) = (E[p(A→B)] - E[p(B→A)]) / (E[p(A→B)] + E[p(B→A)])
    Range: (-1, 1).  >0 means A→B is dominant direction.
    """
    alpha_ab, beta_ab = get_prior(a, b)
    alpha_ba, beta_ba = get_prior(b, a)

    e_ab = posterior_mean(f_ab, f_ba, alpha_ab, beta_ab)
    e_ba = posterior_mean(f_ba, f_ab, alpha_ba, beta_ba)

    denom = e_ab + e_ba
    if denom == 0:
        return 0.0
    return (e_ab - e_ba) / denom



# ──────────────────────────────────────────────────────────────────────────────
# SECTION 4 — count directly-follows pairs from the event log
# ──────────────────────────────────────────────────────────────────────────────

def build_directly_follows(event_log: pd.DataFrame) -> dict:
    """
    event_log must have columns: case_id, activity, timestamp.
    Returns dict  (A, B) -> count  for directly-follows pairs.
    """
    #event_log['activity_order'] = event_log['activity'].map(activity_order)
    df = (event_log[['ID_seq', 'activity', 'timestamp', 'activity_order']]
          .sort_values(['ID_seq', 'timestamp', 'activity_order']))

    counts = defaultdict(int)
    start_counts = defaultdict(int)
    end_counts   = defaultdict(int)

    for _, group in df.groupby('ID_seq', sort=False):
        acts = group['activity'].tolist()
        start_counts[acts[0]]  += 1
        end_counts[acts[-1]]   += 1
        for i in range(len(acts) - 1):
            counts[(acts[i], acts[i + 1])] += 1

    return dict(counts), dict(start_counts), dict(end_counts)

    # ──────────────────────────────────────────────────────────────────────────────
# SECTION 5 — build dependency matrix and filter edges
# ──────────────────────────────────────────────────────────────────────────────

def compute_dependency_matrix(df_counts: dict) -> dict:
    """
    For every observed pair (A,B) compute dep_bayes(A,B).
    """
    dep = {}
    all_pairs = set(df_counts.keys())
    for (a, b) in all_pairs:
        f_ab = df_counts.get((a, b), 0)
        f_ba = df_counts.get((b, a), 0)
        dep[(a, b)] = bayesian_dependency(a, b, f_ab, f_ba)
    return dep


def filter_edges(dep_matrix: dict,
                 df_counts:  dict,
                 dep_threshold: float = 0.5,
                 min_freq:       int   = 10) -> list:
    """
    Keep edge (A,B) if:
      - dep_bayes(A,B) >= dep_threshold
      - raw count f(A,B) >= min_freq
    Returns list of (A, B, dep_score, freq).
    """
    edges = []
    for (a, b), score in dep_matrix.items():
        freq = df_counts.get((a, b), 0)
        if score >= dep_threshold and freq >= min_freq:
            edges.append((a, b, score, freq))
    return edges

    
# ──────────────────────────────────────────────────────────────────────────────
# SECTION 6 — visualise with graphviz (same look as pm4py heu-net)
# ──────────────────────────────────────────────────────────────────────────────

def render_heu_net(edges: list,
                   start_counts: dict,
                   end_counts:   dict,
                   may_31_output_file:  str = "bayesian_heu_net") -> None:
    """
    Render the Bayesian Heuristic Net as a PNG using graphviz.
    Edge labels show  dep_bayes score and raw frequency.
    """
    dot = Digraph(name="Bayesian Heuristic Net",
                  graph_attr={'rankdir': 'TB', 'fontname': 'Helvetica'},
                  node_attr={'shape': 'rectangle', 'style': 'filled',
                             'fillcolor': '#5f9ea0', 'fontcolor': 'white',
                             'fontname': 'Helvetica'},
                  edge_attr={'fontname': 'Helvetica', 'fontsize': '9'})

    # Start / end tokens
    dot.node('__start__', '', shape='ellipse', fillcolor='green', style='filled')
    dot.node('__end__',   '', shape='ellipse', fillcolor='orange', style='filled')

    # Activity nodes — collect unique activities from edges
    activities = set()
    for (a, b, *_) in edges:
        activities.add(a)
        activities.add(b)

    # Also add start/end linked activities
    start_acts = {a for a, cnt in start_counts.items() if a in activities}
    end_acts   = {a for a, cnt in end_counts.items()   if a in activities}

    total_cases = sum(start_counts.values()) or 1

    for act in activities:
        dot.node(act, act)

    # Start / end arcs
    for act in start_acts:
        freq = start_counts.get(act, 0)
        dot.edge('__start__', act, label=str(freq))
    for act in end_acts:
        freq = end_counts.get(act, 0)
        dot.edge(act, '__end__', label=str(freq))

    # Dependency edges
    for (a, b, score, freq) in edges:
        label = f"{freq}\n({score:.3f})"
        dot.edge(a, b, label=label)

    dot.render(may_31_output_file, format='png', cleanup=True)
    print(f"Saved: {may_31_output_file}.png")

    # ──────────────────────────────────────────────────────────────────────────────
# SECTION 7 — top-level entry point (mirrors your original create_heuristic_net)
# ──────────────────────────────────────────────────────────────────────────────

def create_bayesian_heuristic_net(
        data:              pd.DataFrame,
        surgery_name:      str,
        timestamp_columns: list,
        dep_threshold:     float = 0.5,   # ← lower = more edges; tune as needed
        min_freq:          int   = 10,    # ← minimum raw count to keep an edge
) -> None:
    """
    Drop-in replacement for create_heuristic_net().

    Parameters
    ----------
    data               : raw dataframe with ID_seq + timestamp columns
    surgery_name       : used for output filename
    timestamp_columns  : list of raw column names to melt
    dep_threshold      : Bayesian dependency score threshold  (0–1)
    min_freq           : minimum directly-follows count to keep edge
    """
    # ── count directly-follows ───────────────────────────────────────────────
    df_counts, start_counts, end_counts = build_directly_follows(df_eventlog)

    # ── Bayesian dependency matrix ───────────────────────────────────────────
    dep_matrix = compute_dependency_matrix(df_counts)

    # ── filter ───────────────────────────────────────────────────────────────
    edges = filter_edges(dep_matrix, df_counts,
                         dep_threshold=dep_threshold,
                         min_freq=min_freq)

    print(f"Edges retained after Bayesian filtering: {len(edges)}")

    # ── render ───────────────────────────────────────────────────────────────
    may_31_output_file = f"{surgery_name}_bayesian_heu_net"
    render_heu_net(edges, start_counts, end_counts, may_31_output_file)

    # ── optional: print dependency table ─────────────────────────────────────
    rows = [(a, b, round(score, 4), freq) for (a, b, score, freq) in
            sorted(edges, key=lambda x: -x[2])]
    print("\nTop Bayesian dependencies:")
    print(f"{'A':<22} {'B':<22} {'dep_bayes':>10} {'freq':>8}")
    print("-" * 66)
    for a, b, score, freq in rows:
        print(f"{a:<22} {b:<22} {score:>10.4f} {freq:>8}")

        # ──────────────────────────────────────────────────────────────────────────────
# SECTION 8 — run
# ──────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    data = event_log

    timestamp_columns = list(activity_mapping.keys())

    create_bayesian_heuristic_net(
        data=data,                        # your dataframe
        surgery_name='ALL_SURG',
        timestamp_columns=timestamp_columns,
        dep_threshold=0.5,                # tune with domain experts
        min_freq=10,  
    )


C:\Users\hfarheen\AppData\Local\Temp\ipykernel_2392\1059658092.py:3: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("Perioperative_Dataset_csv.csv")


Edges retained after Bayesian filtering: 32
Saved: ALL_SURG_bayesian_heu_net.png

Top Bayesian dependencies:
A                      B                       dep_bayes     freq
------------------------------------------------------------------
Incision               Dressing                   1.0000   161270
Patient Ready          Prep Start                 1.0000   160888
Recovery Out           Discharge                  1.0000   160875
Prep Start             Incision                   1.0000   160865
Admission              Patient Sent               1.0000   138444
Induction Start        Patient Ready              1.0000   132661
Patient In             Induction Start            1.0000   132658
Cleaning End           Patient In                 1.0000   132605
Cleaning Start         Cleaning End               1.0000   132569
Recovery In            Recovery Out               1.0000   160875
Patient In             Patient Ready              0.9999    28203
Admission              Patient A